In [2]:
import ee

ee.Authenticate()
ee.Initialize()

Enter verification code:  4/1AeoWuM-yLhliQM4GDTNLqeMZvQLYLFEpS9uHq4xyFScDoxCqncl_CDPkCes



Successfully saved authorization token.


import ee
ee.Initialize()

states = ee.FeatureCollection("TIGER/2018/States")
colorado = states.filter(ee.Filter.eq("NAME", "Colorado")).geometry()

mtbs = ee.FeatureCollection("USFS/GTAC/MTBS/burned_area_boundaries/v1")

print("MTBS properties:")
print(mtbs.first().propertyNames().getInfo())

print("Total MTBS in Colorado:")
print(mtbs.filterBounds(colorado).size().getInfo())

sample = mtbs.filterBounds(colorado).limit(5)
print(sample.getInfo())

import ee
ee.Initialize()

# ---------------------------------------------------
# Colorado boundary
# ---------------------------------------------------
states = ee.FeatureCollection("TIGER/2018/States")

colorado = states.filter(
    ee.Filter.eq("NAME", "Colorado")
).geometry()

# ---------------------------------------------------
# MTBS collection
# ---------------------------------------------------
mtbs = ee.FeatureCollection(
    "USFS/GTAC/MTBS/burned_area_boundaries/v1"
)

# ---------------------------------------------------
# Inspect fields
# ---------------------------------------------------
first = mtbs.first()

print("Property names:")
print(first.propertyNames().getInfo())

print("\nFirst feature:")
print(first.toDictionary().getInfo())

possible_fields = [
    "Ig_Year",
    "Fire_Year",
    "YEAR",
    "Year",
    "FIRE_YEAR",
    "Incid_Year"
]

for field in possible_fields:
    try:
        n2020 = (
            mtbs
            .filterBounds(colorado)
            .filter(ee.Filter.eq(field, 2020))
            .size()
            .getInfo()
        )

        n2019 = (
            mtbs
            .filterBounds(colorado)
            .filter(ee.Filter.eq(field, 2019))
            .size()
            .getInfo()
        )

        print(f"{field}: 2019={n2019}, 2020={n2020}")

    except Exception as e:
        print(field, "failed")

# Getting data from GEE

import ee
import geemap
import pandas as pd

ee.Initialize()

# --------------------------------------------------
# Parameters
# --------------------------------------------------
FIRE_YEAR = 2020
BEFORE_YEAR = 2019
AFTER_YEAR = 2021
N_FIRES = 20
OUT_CSV = f"CO_{FIRE_YEAR}_MTBS_{N_FIRES}_AEF_GRIDMET_training_table.csv"

# --------------------------------------------------
# Colorado boundary
# --------------------------------------------------
states = ee.FeatureCollection("TIGER/2018/States")
colorado = states.filter(ee.Filter.eq("NAME", "Colorado")).geometry()

# --------------------------------------------------
# MTBS fire perimeters
# --------------------------------------------------
mtbs = ee.FeatureCollection("USFS/GTAC/MTBS/burned_area_boundaries/v1")

print("MTBS property names:")
print(mtbs.first().propertyNames().getInfo())

FIRE_DATE_FIELD = "Ig_Date"

# --------------------------------------------------
# Add fire year from Ig_Date
# --------------------------------------------------
def add_fire_year_from_date(f):
    fire_date = ee.Date(f.get(FIRE_DATE_FIELD))

    return f.set({
        "fire_date": fire_date.format("YYYY-MM-dd"),
        "fire_year_from_date": fire_date.get("year")
    })

mtbs_with_year = mtbs.map(add_fire_year_from_date)

# --------------------------------------------------
# Filter Colorado 2020 fires
# --------------------------------------------------
fires = (
    mtbs_with_year
    .filterBounds(colorado)
    .filter(ee.Filter.eq("fire_year_from_date", FIRE_YEAR))
    .limit(N_FIRES)
)

print("Number of selected Colorado 2020 fires:")
print(fires.size().getInfo())

print("First selected fire:")
print(fires.first().toDictionary().getInfo())

# --------------------------------------------------
# Add area and centroid
# --------------------------------------------------
def add_area(f):
    area_ha = f.geometry().area(maxError=30).divide(10000)
    centroid = f.geometry().centroid(maxError=30).coordinates()

    return f.set({
        "event_id": ee.String("fire_").cat(ee.String(f.id())),
        "fire_year": FIRE_YEAR,
        "burn_area_ha": area_ha,
        "lon": centroid.get(0),
        "lat": centroid.get(1)
    })

fires = fires.map(add_area)

# --------------------------------------------------
# AEF / Satellite Embeddings
# --------------------------------------------------
emb = ee.ImageCollection("GOOGLE/SATELLITE_EMBEDDING/V1/ANNUAL")

emb_before_raw = emb.filterDate(
    f"{BEFORE_YEAR}-01-01",
    f"{BEFORE_YEAR}-12-31"
).first()

emb_after_raw = emb.filterDate(
    f"{AFTER_YEAR}-01-01",
    f"{AFTER_YEAR}-12-31"
).first()

# Force safe projection for Colorado extraction
emb_before = (
    emb_before_raw
    .resample("bilinear")
    .reproject(crs="EPSG:4326", scale=30)
)

emb_after = (
    emb_after_raw
    .resample("bilinear")
    .reproject(crs="EPSG:4326", scale=30)
)

bands = emb_before.bandNames()

before = emb_before.rename(
    bands.map(lambda b: ee.String("before_").cat(ee.String(b)))
)

after = emb_after.rename(
    bands.map(lambda b: ee.String("after_").cat(ee.String(b)))
)

delta = emb_after.subtract(emb_before).rename(
    bands.map(lambda b: ee.String("delta_").cat(ee.String(b)))
)

embedding_stack = (
    before
    .addBands(after)
    .addBands(delta)
    .reproject(crs="EPSG:4326", scale=30)
)

# --------------------------------------------------
# gridMET fire-weather extraction
# --------------------------------------------------
gridmet = ee.ImageCollection("IDAHO_EPSCOR/GRIDMET")

def add_fire_weather(f):
    fire_date = ee.Date(f.get(FIRE_DATE_FIELD))

    window = gridmet.filterDate(
        fire_date.advance(-3, "day"),
        fire_date.advance(4, "day")
    )

    vpd_day = (
        gridmet
        .filterDate(fire_date, fire_date.advance(1, "day"))
        .select("vpd")
        .mean()
    )

    vpd7 = window.select("vpd").mean()
    tmax7 = window.select("tmmx").mean()
    pr7 = window.select("pr").sum()

    def zonal_mean(img, band):
        return img.reduceRegion(
            reducer=ee.Reducer.mean(),
            geometry=f.geometry(),
            scale=4000,
            crs="EPSG:4326",
            maxPixels=1e9,
            bestEffort=True,
            tileScale=4
        ).get(band)

    return f.set({
        "vpd_fire_day": zonal_mean(vpd_day, "vpd"),
        "vpd_7day": zonal_mean(vpd7, "vpd"),
        "tmmx_7day": zonal_mean(tmax7, "tmmx"),
        "pr_7day": zonal_mean(pr7, "pr")
    })

fires = fires.map(add_fire_weather)

# --------------------------------------------------
# Extract mean embeddings over fire polygons
# --------------------------------------------------
training = embedding_stack.reduceRegions(
    collection=fires,
    reducer=ee.Reducer.mean(),
    scale=30,
    crs="EPSG:4326",
    tileScale=8
)

# --------------------------------------------------
# Export directly to CSV
# --------------------------------------------------
geemap.ee_to_csv(training, OUT_CSV)

print(f"Saved: {OUT_CSV}")

In [11]:
df = pd.read_csv("CO_2020_MTBS_20_AEF_GRIDMET_training_table.csv")

print(df.columns.tolist())

['Asmnt_Type', 'BurnBndAc', 'BurnBndLat', 'BurnBndLon', 'Comment', 'Event_ID', 'High_T', 'Ig_Date', 'IncGreen_T', 'Incid_Name', 'Incid_Type', 'Low_T', 'Map_ID', 'Map_Prog', 'Mod_T', 'NoData_T', 'Perim_ID', 'Post_ID', 'Pre_ID', 'burn_area_ha', 'dNBR_offst', 'dNBR_stdDv', 'event_id', 'fire_date', 'fire_year', 'fire_year_from_date', 'irwinID', 'lat', 'lon', 'pr_7day', 'tmmx_7day', 'vpd_7day', 'vpd_fire_day']


import ee
import geemap
import pandas as pd

ee.Initialize()

# --------------------------------------------------
# Parameters
# --------------------------------------------------
FIRE_YEAR = 2020
BEFORE_YEAR = 2019
AFTER_YEAR = 2021
N_FIRES = 20
BUFFER_M = 1000

OUT_CSV = f"CO_{FIRE_YEAR}_MTBS_{N_FIRES}_AEF_GRIDMET_BUFFER_training_table.csv"

# --------------------------------------------------
# Colorado and MTBS
# --------------------------------------------------
states = ee.FeatureCollection("TIGER/2018/States")
colorado = states.filter(ee.Filter.eq("NAME", "Colorado")).geometry()

mtbs = ee.FeatureCollection("USFS/GTAC/MTBS/burned_area_boundaries/v1")

FIRE_DATE_FIELD = "Ig_Date"

print("MTBS properties:")
print(mtbs.first().propertyNames().getInfo())

# --------------------------------------------------
# Add fire year from Ig_Date
# --------------------------------------------------
def add_fire_year_from_date(f):
    fire_date = ee.Date(f.get(FIRE_DATE_FIELD))

    return f.set({
        "fire_date": fire_date.format("YYYY-MM-dd"),
        "fire_year_from_date": fire_date.get("year")
    })

mtbs_with_year = mtbs.map(add_fire_year_from_date)

# --------------------------------------------------
# Filter Colorado 2020 fires
# --------------------------------------------------
fires_raw = (
    mtbs_with_year
    .filterBounds(colorado)
    .filter(ee.Filter.eq("fire_year_from_date", FIRE_YEAR))
    .limit(N_FIRES)
)

print("Selected Colorado 2020 fires:")
print(fires_raw.size().getInfo())

print("First selected fire:")
print(fires_raw.first().toDictionary().getInfo())

# --------------------------------------------------
# Convert MTBS polygons to clean centroid buffers
# --------------------------------------------------
def make_centroid_buffer(f):
    fire_date = ee.Date(f.get(FIRE_DATE_FIELD))

    lon = ee.Number.parse(f.get("BurnBndLon"))
    lat = ee.Number.parse(f.get("BurnBndLat"))

    point = ee.Geometry.Point([lon, lat])
    buffer_geom = point.buffer(BUFFER_M)

    # BurnBndAc is acres, convert to hectares
    burn_area_ha = ee.Number(f.get("BurnBndAc")).multiply(0.404686)

    return ee.Feature(buffer_geom).copyProperties(f).set({
        "event_id": ee.String("fire_").cat(ee.String(f.id())),
        "fire_year": FIRE_YEAR,
        "fire_date": fire_date.format("YYYY-MM-dd"),
        "burn_area_ha": burn_area_ha,
        "lon": lon,
        "lat": lat,
        "buffer_m": BUFFER_M
    })

fires = fires_raw.map(make_centroid_buffer)

# --------------------------------------------------
# gridMET weather extraction
# --------------------------------------------------
gridmet = ee.ImageCollection("IDAHO_EPSCOR/GRIDMET")

def add_fire_weather(f):
    fire_date = ee.Date(f.get(FIRE_DATE_FIELD))

    window = gridmet.filterDate(
        fire_date.advance(-3, "day"),
        fire_date.advance(4, "day")
    )

    vpd_day = (
        gridmet
        .filterDate(fire_date, fire_date.advance(1, "day"))
        .select("vpd")
        .mean()
    )

    vpd7 = window.select("vpd").mean()
    tmax7 = window.select("tmmx").mean()
    pr7 = window.select("pr").sum()

    def zonal_mean(img, band):
        return img.reduceRegion(
            reducer=ee.Reducer.mean(),
            geometry=f.geometry(),
            scale=4000,
            maxPixels=1e9,
            bestEffort=True,
            tileScale=4
        ).get(band)

    return f.set({
        "vpd_fire_day": zonal_mean(vpd_day, "vpd"),
        "vpd_7day": zonal_mean(vpd7, "vpd"),
        "tmmx_7day": zonal_mean(tmax7, "tmmx"),
        "pr_7day": zonal_mean(pr7, "pr")
    })

fires = fires.map(add_fire_weather)

# --------------------------------------------------
# AEF / Satellite Embeddings
# --------------------------------------------------
emb = ee.ImageCollection("GOOGLE/SATELLITE_EMBEDDING/V1/ANNUAL")

emb_before = emb.filterDate(
    f"{BEFORE_YEAR}-01-01",
    f"{BEFORE_YEAR}-12-31"
).first()

emb_after = emb.filterDate(
    f"{AFTER_YEAR}-01-01",
    f"{AFTER_YEAR}-12-31"
).first()

print("Before embedding band count:")
print(emb_before.bandNames().size().getInfo())

print("Before embedding first 10 bands:")
print(emb_before.bandNames().slice(0, 10).getInfo())

bands = emb_before.bandNames()

before = emb_before.select(
    bands,
    bands.map(lambda b: ee.String("before_").cat(ee.String(b)))
)

after = emb_after.select(
    bands,
    bands.map(lambda b: ee.String("after_").cat(ee.String(b)))
)

delta = emb_after.subtract(emb_before).select(
    bands,
    bands.map(lambda b: ee.String("delta_").cat(ee.String(b)))
)

embedding_stack = before.addBands(after).addBands(delta)

print("Embedding stack band count:")
print(embedding_stack.bandNames().size().getInfo())

print("Embedding stack first 10 bands:")
print(embedding_stack.bandNames().slice(0, 10).getInfo())

# --------------------------------------------------
# Test embedding extraction on first centroid buffer
# --------------------------------------------------
test_extract = embedding_stack.reduceRegion(
    reducer=ee.Reducer.mean(),
    geometry=fires.first().geometry(),
    scale=100,
    maxPixels=1e9,
    bestEffort=True,
    tileScale=16
)

test_dict = test_extract.getInfo()

print("Test extraction first keys:")
print(list(test_dict.keys())[:20])

print("Number of extracted embedding values:")
print(len(test_dict))

if len(test_dict) == 0:
    raise ValueError("Embedding extraction failed. Try larger BUFFER_M or different scale.")

# --------------------------------------------------
# Extract embeddings for all fires
# --------------------------------------------------
training = embedding_stack.reduceRegions(
    collection=fires,
    reducer=ee.Reducer.mean(),
    scale=100,
    tileScale=16
)

# --------------------------------------------------
# Check output before export
# --------------------------------------------------
first_out = training.first().toDictionary().getInfo()

embedding_out_cols = [
    k for k in first_out.keys()
    if k.startswith("before_") or k.startswith("after_") or k.startswith("delta_")
]

print("Number of embedding columns in first output:")
print(len(embedding_out_cols))

print("First 10 embedding output columns:")
print(embedding_out_cols[:10])

if len(embedding_out_cols) == 0:
    raise ValueError("No embedding columns in output.")

# --------------------------------------------------
# Export local CSV
# --------------------------------------------------
geemap.ee_to_csv(training, OUT_CSV)

print(f"Saved: {OUT_CSV}")

# --------------------------------------------------
# Verify local CSV
# --------------------------------------------------
df_check = pd.read_csv(OUT_CSV)

embedding_cols_local = [
    c for c in df_check.columns
    if c.startswith("before_") or c.startswith("after_") or c.startswith("delta_")
]

print("Local CSV shape:", df_check.shape)
print("Embedding columns in local CSV:", len(embedding_cols_local))
print("First 10 embedding columns:")
print(embedding_cols_local[:10])

In [24]:
# ============================================================
# FULL WORKFLOW:
# MTBS fire/non-fire labeled points + topography + climate/VPD
# + embedding pre-fire/post-fire summaries for XGBoost
# ============================================================

import ee
import pandas as pd
import os

ee.Initialize()

# ============================================================
# 0. SETTINGS
# ============================================================

SAFE_CRS = "EPSG:4326"
SCALE = 100
SEED = 42

START_YEAR = 2000
END_YEAR = 2023
N_FIRES = 20

NONFIRE_BUFFER_M = 1135   # approx. 1000-acre radius
INNER_BUFFER_M = 200      # avoid placing non-fire too close to fire center

OUT_CSV = "xgboost_fire_nonfire_climate_topo_embedding.csv"


# ============================================================
# 1. STUDY AREA
# ============================================================

states = ee.FeatureCollection("TIGER/2018/States")

study_area = states.filter(
    ee.Filter.eq("NAME", "Colorado")
).geometry()


# ============================================================
# 2. LOAD AND FILTER MTBS
# ============================================================

mtbs_all = ee.FeatureCollection(
    "USFS/GTAC/MTBS/burned_area_boundaries/v1"
)

mtbs_bounds = mtbs_all.filterBounds(study_area)

print("MTBS in study area:", mtbs_bounds.size().getInfo())


def clean_fire(f):
    fire_year = ee.Date(f.get("Ig_Date")).get("year")

    geom = (
        f.geometry()
        .transform(SAFE_CRS, 1)
        .buffer(0, 1)
        .simplify(30)
    )

    return ee.Feature(geom, {
        "fire_id": f.get("Event_ID"),
        "fire_name": f.get("Incid_Name"),
        "fire_year": fire_year
    })


fires_clean_all = mtbs_bounds.map(clean_fire)

fires_clean = (
    fires_clean_all
    .filter(ee.Filter.gte("fire_year", START_YEAR))
    .filter(ee.Filter.lte("fire_year", END_YEAR))
    .randomColumn("rand", SEED)
    .sort("rand")
    .limit(N_FIRES)
)

print("Selected fires:", fires_clean.size().getInfo())


# ============================================================
# 3. CREATE FIRE POINTS
# label = 1
# Uses centroid to avoid random-point failure inside invalid polygons
# ============================================================

def make_fire_point(f):
    pt = f.geometry().centroid(30)

    return ee.Feature(pt, {
        "fire_id": f.get("fire_id"),
        "fire_name": f.get("fire_name"),
        "fire_year": f.get("fire_year"),
        "label": 1,
        "sample_type": "fire"
    })


fire_points = fires_clean.map(make_fire_point)

print("Fire points:", fire_points.size().getInfo())


# ============================================================
# 4. CREATE NON-FIRE POINTS
# label = 0
# Safer method: make a nearby point from centroid offset
# This avoids randomPoints failing on empty ring geometries.
# ============================================================

def make_nonfire_point(f):
    center = f.geometry().centroid(30)

    # Make deterministic offset point near the fire.
    # 0.02 degrees is roughly 2 km, enough for nearby non-fire.
    coords = center.coordinates()
    lon = ee.Number(coords.get(0))
    lat = ee.Number(coords.get(1))

    nonfire_geom = ee.Geometry.Point([
        lon.add(0.02),
        lat.add(0.02)
    ])

    # If shifted point falls outside study area, use opposite direction
    alt_geom = ee.Geometry.Point([
        lon.subtract(0.02),
        lat.subtract(0.02)
    ])

    nonfire_geom = ee.Algorithms.If(
        study_area.contains(nonfire_geom, 30),
        nonfire_geom,
        alt_geom
    )

    return ee.Feature(ee.Geometry(nonfire_geom), {
        "fire_id": f.get("fire_id"),
        "fire_name": f.get("fire_name"),
        "fire_year": f.get("fire_year"),
        "label": 0,
        "sample_type": "non_fire"
    })


nonfire_points = fires_clean.map(make_nonfire_point)

print("Non-fire points:", nonfire_points.size().getInfo())


# ============================================================
# 5. COMBINE POINTS
# ============================================================

samples = fire_points.merge(nonfire_points)

print("Total sample points:", samples.size().getInfo())


# ============================================================
# 6. TOPOGRAPHY PREDICTORS
# ============================================================

dem = ee.Image("USGS/SRTMGL1_003").select("elevation").rename("elevation")
slope = ee.Terrain.slope(dem).rename("slope")
aspect = ee.Terrain.aspect(dem).rename("aspect")

topo_img = dem.addBands(slope).addBands(aspect)


# ============================================================
# 7. CLIMATE / VPD PREDICTORS
# TerraClimate monthly
# Units:
# tmmx/tmmn are temperature * 0.1 C
# vpd is vapor pressure deficit * 0.01 kPa
# ppt is mm
# ============================================================

terraclimate = ee.ImageCollection("IDAHO_EPSCOR/TERRACLIMATE")


def climate_for_year(year, prefix):
    year = ee.Number(year)

    start = ee.Date.fromYMD(year, 1, 1)
    end = start.advance(1, "year")

    col = terraclimate.filterDate(start, end)

    ppt = col.select("pr").sum().rename(prefix + "_ppt_sum")

    tmax = col.select("tmmx").mean().multiply(0.1).rename(prefix + "_tmax_mean")
    tmin = col.select("tmmn").mean().multiply(0.1).rename(prefix + "_tmin_mean")

    vpd = col.select("vpd").mean().multiply(0.01).rename(prefix + "_vpd_mean")

    return ppt.addBands(tmax).addBands(tmin).addBands(vpd)


# ============================================================
# 8. EMBEDDING PREDICTORS
# Replace collection name if needed.
# This assumes your embedding image collection has many bands.
# ============================================================

embedding_ic = ee.ImageCollection("GOOGLE/SATELLITE_EMBEDDING/V1/ANNUAL")


def rename_embedding(img, prefix):
    old_names = img.bandNames()

    new_names = old_names.map(
        lambda b: ee.String(prefix).cat("_").cat(ee.String(b))
    )

    return img.rename(new_names)


def embedding_for_year(year, prefix):
    year = ee.Number(year)

    start = ee.Date.fromYMD(year, 1, 1)
    end = start.advance(1, "year")

    img = (
        embedding_ic
        .filterDate(start, end)
        .filterBounds(study_area)
        .mosaic()
        .toFloat()
    )

    return rename_embedding(img, prefix)


# ============================================================
# 9. CREATE PER-POINT PREDICTOR IMAGE AND SAMPLE
# Important: predictors depend on each point's fire_year.
# Therefore map over each point individually.
# ============================================================

def sample_one_point(f):
    fire_year = ee.Number(f.get("fire_year"))

    pre_year = fire_year.subtract(1)
    post_year = fire_year.add(1)

    climate_pre = climate_for_year(pre_year, "pre")
    climate_post = climate_for_year(post_year, "post")

    emb_pre = embedding_for_year(pre_year, "pre_emb")
    emb_post = embedding_for_year(post_year, "post_emb")

    # Delta embedding = post - pre
    # Need rename back temporarily for subtraction
    raw_pre = embedding_for_year(pre_year, "tmp_pre")
    raw_post = embedding_for_year(post_year, "tmp_post")

    # Simpler delta: subtract original annual mosaics before renaming
    pre_raw = (
        embedding_ic
        .filterDate(ee.Date.fromYMD(pre_year, 1, 1),
                    ee.Date.fromYMD(pre_year.add(1), 1, 1))
        .filterBounds(study_area)
        .mosaic()
        .toFloat()
    )

    post_raw = (
        embedding_ic
        .filterDate(ee.Date.fromYMD(post_year, 1, 1),
                    ee.Date.fromYMD(post_year.add(1), 1, 1))
        .filterBounds(study_area)
        .mosaic()
        .toFloat()
    )

    delta_emb = rename_embedding(
        post_raw.subtract(pre_raw),
        "delta_emb"
    )

    predictor_img = (
        topo_img
        .addBands(climate_pre)
        .addBands(climate_post)
        .addBands(emb_pre)
        .addBands(emb_post)
        .addBands(delta_emb)
        .toFloat()
        .unmask(-9999)
        .setDefaultProjection(SAFE_CRS, None, SCALE)
    )

    sampled = predictor_img.sampleRegions(
        collection=ee.FeatureCollection([f]),
        properties=[
            "fire_id",
            "fire_name",
            "fire_year",
            "label",
            "sample_type"
        ],
        scale=SCALE,
        projection=SAFE_CRS,
        geometries=True,
        tileScale=16
    ).first()

    return sampled


training = samples.map(sample_one_point)

print("Training rows expected:", samples.size().getInfo())


# ============================================================
# 10. SAVE LOCALLY TO CURRENT PYTHON FOLDER
# ============================================================

training_dict = training.getInfo()
features = training_dict.get("features", [])

print("Features returned:", len(features))

rows = []

for feat in features:
    props = feat.get("properties", {}).copy()

    geom = feat.get("geometry", None)
    if geom is not None:
        coords = geom["coordinates"]
        props["longitude"] = coords[0]
        props["latitude"] = coords[1]

    rows.append(props)

df = pd.DataFrame(rows)

df.to_csv(OUT_CSV, index=False)

print("Saved CSV here:")
print(os.path.abspath(OUT_CSV))

print("Rows saved:", len(df))

if len(df) > 0:
    print("Label counts:")
    print(df["label"].value_counts())

    embedding_cols = [
        c for c in df.columns
        if c.startswith("pre_emb_")
        or c.startswith("post_emb_")
        or c.startswith("delta_emb_")
    ]

    print("Embedding columns:", len(embedding_cols))
    display(df.head())
else:
    print("No rows returned.")

MTBS in study area: 456
Selected fires: 20
Fire points: 20
Non-fire points: 20
Total sample points: 40
Training rows expected: 40
Features returned: 40
Saved CSV here:
/home/jovyan/data-store/Nayani/xgboost_fire_nonfire_climate_topo_embedding.csv
Rows saved: 40
Label counts:
label
1    20
0    20
Name: count, dtype: int64
Embedding columns: 192


,aspect,elevation,fire_id,fire_name,fire_year,label,post_ppt_sum,post_tmax_mean,post_tmin_mean,post_vpd_mean,...,pre_emb_A54,pre_emb_A55,pre_emb_A56,pre_emb_A57,pre_emb_A58,pre_emb_A59,pre_emb_A60,pre_emb_A61,pre_emb_A62,pre_emb_A63
0,138.806778,2685,CO3888410493320120623,WALDO CANYON,2012,1,431,11.308333,-4.866667,0.537500,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,270.000000,1586,CO3849910432220081002,HAYNES CREEK,2008,1,338,19.191668,1.641667,0.996667,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,90.000000,1328,CO3975410279220221022,CR 15,2022,1,529,18.975000,2.358333,0.856667,...,0.048228,0.035433,-0.2599,-0.147697,-0.062991,0.032541,-0.088827,0.113741,0.098424,0.093564
3,333.340118,3038,CO3999910724920020719,BIG FISH,2002,1,496,8.700000,-6.491667,0.510000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,0.000000,1171,CO3800010300020110407,FT. LYONS,2011,1,182,23.766666,3.958333,1.355000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [25]:
# ============================================================
# FULL WORKFLOW:
# 2020 MTBS fire points = 1
# nearby non-fire points = 0
# topography + climate/VPD + satellite embeddings
# save CSV locally
# ============================================================

import ee
import pandas as pd
import os

ee.Initialize()

# ============================================================
# 0. SETTINGS
# ============================================================

SAFE_CRS = "EPSG:4326"
SCALE = 100
SEED = 42

FIRE_YEAR = 2020
PRE_YEAR = 2019
POST_YEAR = 2021

N_FIRES = 20
NONFIRE_OFFSET_DEG = 0.02   # roughly 2 km
OUT_CSV = "xgboost_fire_nonfire_2020.csv"


# ============================================================
# 1. STUDY AREA
# ============================================================

states = ee.FeatureCollection("TIGER/2018/States")

study_area = states.filter(
    ee.Filter.eq("NAME", "Colorado")
).geometry()


# ============================================================
# 2. LOAD 2020 MTBS FIRES
# ============================================================

mtbs_all = ee.FeatureCollection(
    "USFS/GTAC/MTBS/burned_area_boundaries/v1"
)

mtbs_bounds = mtbs_all.filterBounds(study_area)

print("MTBS in study area:", mtbs_bounds.size().getInfo())


def clean_fire(f):
    fire_year = ee.Date(f.get("Ig_Date")).get("year")

    geom = (
        f.geometry()
        .transform(SAFE_CRS, 1)
        .buffer(0, 1)
        .simplify(30)
    )

    return ee.Feature(geom, {
        "fire_id": f.get("Event_ID"),
        "fire_name": f.get("Incid_Name"),
        "fire_year": fire_year
    })


fires_clean_all = mtbs_bounds.map(clean_fire)

fires_clean = (
    fires_clean_all
    .filter(ee.Filter.eq("fire_year", FIRE_YEAR))
    .randomColumn("rand", SEED)
    .sort("rand")
    .limit(N_FIRES)
)

print("Selected 2020 fires:", fires_clean.size().getInfo())


# ============================================================
# 3. FIRE POINTS
# ============================================================

def make_fire_point(f):
    pt = f.geometry().centroid(30)

    return ee.Feature(pt, {
        "fire_id": f.get("fire_id"),
        "fire_name": f.get("fire_name"),
        "fire_year": f.get("fire_year"),
        "label": 1,
        "sample_type": "fire"
    })


fire_points = fires_clean.map(make_fire_point)

print("Fire points:", fire_points.size().getInfo())


# ============================================================
# 4. NEARBY NON-FIRE POINTS
# ============================================================

def make_nonfire_point(f):
    center = f.geometry().centroid(30)
    coords = center.coordinates()

    lon = ee.Number(coords.get(0))
    lat = ee.Number(coords.get(1))

    candidate_1 = ee.Geometry.Point([
        lon.add(NONFIRE_OFFSET_DEG),
        lat.add(NONFIRE_OFFSET_DEG)
    ])

    candidate_2 = ee.Geometry.Point([
        lon.subtract(NONFIRE_OFFSET_DEG),
        lat.subtract(NONFIRE_OFFSET_DEG)
    ])

    nonfire_geom = ee.Algorithms.If(
        study_area.contains(candidate_1, 30),
        candidate_1,
        candidate_2
    )

    return ee.Feature(ee.Geometry(nonfire_geom), {
        "fire_id": f.get("fire_id"),
        "fire_name": f.get("fire_name"),
        "fire_year": f.get("fire_year"),
        "label": 0,
        "sample_type": "non_fire"
    })


nonfire_points = fires_clean.map(make_nonfire_point)

print("Non-fire points:", nonfire_points.size().getInfo())


# ============================================================
# 5. COMBINE POINTS
# ============================================================

samples = fire_points.merge(nonfire_points)

print("Total sample points:", samples.size().getInfo())


# ============================================================
# 6. TOPOGRAPHY
# ============================================================

dem = ee.Image("USGS/SRTMGL1_003").select("elevation").rename("elevation")
slope = ee.Terrain.slope(dem).rename("slope")
aspect = ee.Terrain.aspect(dem).rename("aspect")

topo_img = dem.addBands(slope).addBands(aspect)


# ============================================================
# 7. CLIMATE / VPD FROM TERRACLIMATE
# ============================================================

terraclimate = ee.ImageCollection("IDAHO_EPSCOR/TERRACLIMATE")


def climate_for_year(year, prefix):
    start = ee.Date.fromYMD(year, 1, 1)
    end = start.advance(1, "year")

    col = terraclimate.filterDate(start, end)

    ppt = col.select("pr").sum().rename(prefix + "_ppt_sum")
    tmax = col.select("tmmx").mean().multiply(0.1).rename(prefix + "_tmax_mean")
    tmin = col.select("tmmn").mean().multiply(0.1).rename(prefix + "_tmin_mean")
    vpd = col.select("vpd").mean().multiply(0.01).rename(prefix + "_vpd_mean")

    return ppt.addBands(tmax).addBands(tmin).addBands(vpd)


climate_pre = climate_for_year(PRE_YEAR, "pre")
climate_post = climate_for_year(POST_YEAR, "post")


# ============================================================
# 8. SATELLITE EMBEDDINGS
# ============================================================

embedding_ic = ee.ImageCollection(
    "GOOGLE/SATELLITE_EMBEDDING/V1/ANNUAL"
)


def raw_embedding_for_year(year):
    start = ee.Date.fromYMD(year, 1, 1)
    end = start.advance(1, "year")

    return (
        embedding_ic
        .filterDate(start, end)
        .filterBounds(study_area)
        .mosaic()
        .toFloat()
        .unmask(-9999)
    )


def rename_embedding(img, prefix):
    old_names = img.bandNames()

    new_names = old_names.map(
        lambda b: ee.String(prefix).cat("_").cat(ee.String(b))
    )

    return img.rename(new_names)


emb_pre_raw = raw_embedding_for_year(PRE_YEAR)
emb_post_raw = raw_embedding_for_year(POST_YEAR)

emb_pre = rename_embedding(emb_pre_raw, "pre_emb")
emb_post = rename_embedding(emb_post_raw, "post_emb")
emb_delta = rename_embedding(emb_post_raw.subtract(emb_pre_raw), "delta_emb")


# ============================================================
# 9. COMBINE ALL PREDICTORS
# ============================================================

predictor_img = (
    topo_img
    .addBands(climate_pre)
    .addBands(climate_post)
    .addBands(emb_pre)
    .addBands(emb_post)
    .addBands(emb_delta)
    .toFloat()
    .unmask(-9999)
    .setDefaultProjection(SAFE_CRS, None, SCALE)
)


# ============================================================
# 10. SAMPLE POINTS
# ============================================================

training = predictor_img.sampleRegions(
    collection=samples,
    properties=[
        "fire_id",
        "fire_name",
        "fire_year",
        "label",
        "sample_type"
    ],
    scale=SCALE,
    projection=SAFE_CRS,
    geometries=True,
    tileScale=16
)


# ============================================================
# 11. SAVE LOCALLY
# ============================================================

training_dict = training.getInfo()
features = training_dict.get("features", [])

print("Features returned:", len(features))

rows = []

for feat in features:
    props = feat.get("properties", {}).copy()

    geom = feat.get("geometry", None)
    if geom is not None:
        coords = geom["coordinates"]
        props["longitude"] = coords[0]
        props["latitude"] = coords[1]

    rows.append(props)

df = pd.DataFrame(rows)

df.to_csv(OUT_CSV, index=False)

print("Saved CSV here:")
print(os.path.abspath(OUT_CSV))
print("Rows saved:", len(df))

if len(df) > 0:
    print("Label counts:")
    print(df["label"].value_counts())

    embedding_cols = [
        c for c in df.columns
        if c.startswith("pre_emb_")
        or c.startswith("post_emb_")
        or c.startswith("delta_emb_")
    ]

    print("Embedding columns:", len(embedding_cols))
    display(df.head())

MTBS in study area: 456
Selected 2020 fires: 20
Fire points: 20
Non-fire points: 20
Total sample points: 40
Features returned: 40
Saved CSV here:
/home/jovyan/data-store/Nayani/xgboost_fire_nonfire_2020.csv
Rows saved: 40
Label counts:
label
1    20
0    20
Name: count, dtype: int64
Embedding columns: 192


,aspect,delta_emb_A00,delta_emb_A01,delta_emb_A02,delta_emb_A03,delta_emb_A04,delta_emb_A05,delta_emb_A06,delta_emb_A07,delta_emb_A08,...,pre_emb_A62,pre_emb_A63,pre_ppt_sum,pre_tmax_mean,pre_tmin_mean,pre_vpd_mean,sample_type,slope,longitude,latitude
0,52.460545,0.005844,0.034879,-0.045952,-0.026144,0.060285,0.055117,-0.042076,-0.169412,0.060531,...,0.059116,0.024606,357,16.458334,-0.508333,0.989167,fire,3.799931,-108.388476,39.787731
1,297.996338,-0.043306,-0.020484,-0.007628,-0.073572,0.019377,0.046813,-0.077570,0.006705,0.022145,...,0.098424,0.017778,558,9.483334,-5.066667,0.527500,fire,8.820938,-106.009738,39.833545
2,167.623322,0.003937,-0.103037,-0.037893,-0.012549,-0.059054,0.084583,-0.049950,-0.006213,0.062561,...,-0.124567,0.035433,384,15.725000,0.225000,0.969167,fire,26.054079,-108.540292,39.431100
3,156.411240,0.029527,0.000000,-0.043737,-0.215609,0.066621,0.131457,-0.035679,-0.137055,0.009350,...,0.113741,-0.041584,585,9.991667,-5.133333,0.545000,fire,4.544702,-106.032195,40.236889
4,69.355591,-0.037093,-0.023437,-0.080523,-0.140254,0.058685,0.054133,-0.034448,-0.100884,0.035986,...,0.130165,-0.013841,523,10.091666,-4.450000,0.541667,fire,2.628833,-106.282825,41.103763


In [14]:
import pandas as pd
import numpy as np
import joblib

from xgboost import XGBRegressor, XGBClassifier
from sklearn.model_selection import LeaveOneOut, cross_val_predict
from sklearn.metrics import mean_absolute_error, r2_score, accuracy_score, roc_auc_score

df = pd.read_csv("CO_2020_fire_nonfire_point_embeddings.csv")

embedding_cols = [
    c for c in df.columns
    if c.startswith("before_")
    or c.startswith("after_")
    or c.startswith("delta_")
]

weather_cols = [
    c for c in ["vpd_summer", "tmmx_summer", "pr_summer"]
    if c in df.columns
]

feature_cols = embedding_cols + weather_cols

X = df[feature_cols].replace([np.inf, -np.inf], np.nan).fillna(0)

# -----------------------------
# Model 1: fire/non-fire classifier
# -----------------------------
y_class = df["class"]

clf = XGBClassifier(
    n_estimators=100,
    learning_rate=0.05,
    max_depth=2,
    subsample=0.85,
    colsample_bytree=0.85,
    eval_metric="logloss",
    random_state=42
)

clf.fit(X, y_class)

df["pred_fire_probability"] = clf.predict_proba(X)[:, 1]

# -----------------------------
# Model 2: affected-area regression
# Non-fire locations have area = 0
# -----------------------------
y_area = np.log1p(df["burn_area_ha"])

reg = XGBRegressor(
    n_estimators=120,
    learning_rate=0.05,
    max_depth=2,
    subsample=0.85,
    colsample_bytree=0.85,
    objective="reg:squarederror",
    random_state=42
)

reg.fit(X, y_area)

df["pred_area_ha"] = np.expm1(reg.predict(X))

print("Training rows:", len(df))
print("Embedding columns:", len(embedding_cols))
print("Mean predicted fire probability:", df["pred_fire_probability"].mean())
print("MAE affected area:", mean_absolute_error(df["burn_area_ha"], df["pred_area_ha"]))

joblib.dump(clf, "xgb_fire_probability_model.pkl")
joblib.dump(reg, "xgb_area_model.pkl")
joblib.dump(feature_cols, "feature_cols.pkl")

df.to_csv("fire_nonfire_training_predictions.csv", index=False)

print("Saved models and predictions.")

ModuleNotFoundError: No module named 'xgboost'

# XGBOOST

In [ ]:
import joblib
import numpy as np
import pandas as pd

from xgboost import XGBRegressor
from sklearn.model_selection import LeaveOneOut, cross_val_predict
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

df = pd.read_csv("CO_2020_MTBS_20_AEF_GRIDMET_training_table.csv")
df = df.dropna(subset=["burn_area_ha"]).copy()

df["log_area"] = np.log1p(df["burn_area_ha"])

feature_cols = [
    c for c in df.columns
    if c.startswith("before_")
    or c.startswith("after_")
    or c.startswith("delta_")
]

feature_cols += [
    c for c in ["vpd_fire_day", "vpd_7day", "tmmx_7day", "pr_7day"]
    if c in df.columns
]

X = df[feature_cols].replace([np.inf, -np.inf], np.nan).fillna(0)
y = df["log_area"]

model = XGBRegressor(
    n_estimators=120,
    learning_rate=0.05,
    max_depth=2,
    subsample=0.85,
    colsample_bytree=0.85,
    objective="reg:squarederror",
    random_state=42
)

loo = LeaveOneOut()
pred_log = cross_val_predict(model, X, y, cv=loo)
pred_area = np.expm1(pred_log)

print("Toy model performance")
print(f"MAE  : {mean_absolute_error(df['burn_area_ha'], pred_area):,.2f} ha")
print(f"RMSE : {mean_squared_error(df['burn_area_ha'], pred_area, squared=False):,.2f} ha")
print(f"R2   : {r2_score(df['burn_area_ha'], pred_area):.3f}")

df["cv_pred_area_ha"] = pred_area

model.fit(X, y)

joblib.dump(model, "xgb_fire_area_toy.pkl")
joblib.dump(feature_cols, "feature_cols.pkl")
df.to_csv("training_with_predictions.csv", index=False)

print("Saved model and training outputs.")

# User interaction

In [ ]:
import ee
import joblib
import numpy as np
import pandas as pd
import streamlit as st

ee.Initialize()

st.set_page_config(
    page_title="Embedding-informed Fire Digital Twin",
    layout="wide"
)

st.title("Embedding-informed fire digital twin demo")

model = joblib.load("xgb_fire_area_toy.pkl")
feature_cols = joblib.load("feature_cols.pkl")
training_df = pd.read_csv("training_with_predictions.csv")

emb = ee.ImageCollection("GOOGLE/SATELLITE_EMBEDDING/V1/ANNUAL")

LATEST_EMBEDDING_YEAR = 2024  # change when newer annual embedding is available
BASELINE_YEAR = 2019

def get_embedding_at_location(lon, lat, buffer_m, year, prefix):
    point = ee.Geometry.Point([lon, lat])
    region = point.buffer(buffer_m)

    img = emb.filterDate(f"{year}-01-01", f"{year}-12-31").first()

    vals = img.reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=region,
        scale=30,
        maxPixels=1e9,
        bestEffort=True
    ).getInfo()

    return {f"{prefix}_{k}": v for k, v in vals.items()}

def build_new_location_row(lon, lat, buffer_m, user_vpd):
    before = get_embedding_at_location(
        lon, lat, buffer_m, BASELINE_YEAR, "before"
    )

    current = get_embedding_at_location(
        lon, lat, buffer_m, LATEST_EMBEDDING_YEAR, "after"
    )

    row = {}
    row.update(before)
    row.update(current)

    # Delta embedding
    before_keys = [k for k in row.keys() if k.startswith("before_")]

    for bk in before_keys:
        band = bk.replace("before_", "")
        ak = f"after_{band}"
        dk = f"delta_{band}"

        if ak in row:
            row[dk] = row[ak] - row[bk]

    # User scenario weather
    row["vpd_fire_day"] = user_vpd
    row["vpd_7day"] = user_vpd

    # Simple defaults for demo if model expects them
    row["tmmx_7day"] = 300
    row["pr_7day"] = 0

    out = pd.DataFrame([row])

    for col in feature_cols:
        if col not in out.columns:
            out[col] = 0

    return out[feature_cols].replace([np.inf, -np.inf], np.nan).fillna(0)

def predict_area(X):
    pred_log = model.predict(X)[0]
    return np.expm1(pred_log)

mode = st.sidebar.radio(
    "Prediction mode",
    ["Existing demo fire", "New user location"]
)

user_vpd = st.sidebar.slider(
    "Current / scenario VPD",
    min_value=0.0,
    max_value=8.0,
    value=2.5,
    step=0.1
)

if mode == "Existing demo fire":
    fire_index = st.sidebar.selectbox(
        "Select training/demo fire",
        training_df.index.tolist()
    )

    row = training_df.loc[[fire_index]].copy()

    if "vpd_7day" in row.columns:
        row["vpd_7day"] = user_vpd
    if "vpd_fire_day" in row.columns:
        row["vpd_fire_day"] = user_vpd

    X = row[feature_cols].replace([np.inf, -np.inf], np.nan).fillna(0)
    pred_area = predict_area(X)

    observed = row["burn_area_ha"].iloc[0]

    col1, col2 = st.columns(2)
    col1.metric("Observed burned area", f"{observed:,.1f} ha")
    col2.metric("Scenario predicted affected area", f"{pred_area:,.1f} ha")

    st.dataframe(row[["burn_area_ha", "vpd_7day", "vpd_fire_day"]])

else:
    lon = st.sidebar.number_input("Longitude", value=-105.25, format="%.6f")
    lat = st.sidebar.number_input("Latitude", value=39.50, format="%.6f")
    buffer_m = st.sidebar.slider(
        "Prediction buffer radius around location",
        min_value=250,
        max_value=5000,
        value=1000,
        step=250
    )

    run = st.sidebar.button("Run new-location digital twin")

    if run:
        with st.spinner("Querying Earth Engine embedding and running model..."):
            X_new = build_new_location_row(
                lon=lon,
                lat=lat,
                buffer_m=buffer_m,
                user_vpd=user_vpd
            )

            pred_area = predict_area(X_new)

        st.subheader("New-location prediction")

        col1, col2, col3 = st.columns(3)
        col1.metric("Predicted affected area", f"{pred_area:,.1f} ha")
        col2.metric("Scenario VPD", f"{user_vpd:.2f}")
        col3.metric("Buffer radius", f"{buffer_m:,} m")

        st.write(
            "This uses the most recent annual embedding as a current-condition proxy "
            "and combines it with the user-provided VPD scenario."
        )

        st.dataframe(X_new)

        st.download_button(
            "Download scenario input/output",
            X_new.assign(predicted_area_ha=pred_area).to_csv(index=False),
            "new_location_digital_twin_prediction.csv",
            "text/csv"
        )
    else:
        st.info("Enter lon/lat and VPD, then run the new-location prediction.")